<a href="https://colab.research.google.com/github/elkins/synth-pdb/blob/main/examples/interactive_tutorials/ubiquitin_rdc_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Validating Synthetic RDCs against Experimental NMR Data
### Using Ubiquitin (PDB: 1D3Z) and Published Restraints

---

## 🎯 Academic Context & The Q-Factor
To prove a computational physics engine is accurately predicting biological observables, we must benchmark it against "gold-standard" experimental data.

In NMR, **Residual Dipolar Couplings (RDCs)** provide highly precise long-range orientational restraints. They describe the angle between an internuclear bond vector (like N-H) and the external magnetic field.

The standard metric for evaluating the agreement between a theoretical 3D structural model and experimental RDCs is the **Cornilescu Q-factor** (*J Biomol NMR*, 1998):

$$ Q = \frac{RMS(D_{exp} - D_{calc})}{RMS(D_{exp})} $$

- **Q < 0.20** indicates excellent agreement (typical of high-resolution structures refined against RDCs).
- **Q < 0.35** indicates a fundamentally correct fold but with local geometric inaccuracies.
- **Q > 0.50** indicates severe structural errors or an incorrect fold.

In this notebook, we will calculate the Q-factor of `synth-pdb`'s internal geometry engine against published experimental data for Ubiquitin, proving our synthetic structural physics are grounded in reality.

In [ ]:
# 🔧 Environment Setup
import os
import sys
import urllib.request

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Running in Google Colab")
    !pip install -q synth-pdb pynmrstar biotite py3dmol matplotlib scipy scikit-learn
else:
    print("💻 Running in local Jupyter environment")
    sys.path.append(os.path.abspath("../../"))

print("✅ Environment configured!")

## 1. Downloading the Experimental Data

We need two pieces of data:
1. **The 3D coordinates** for Ubiquitin (PDB ID: 1D3Z).
2. **The experimental NMR restraints** containing the measured RDCs.

In [ ]:
import subprocess

# Download 1D3Z structure if not present
pdb_file = "1D3Z.pdb"
if not os.path.exists(pdb_file):
    print("Downloading 1D3Z.pdb...")
    urllib.request.urlretrieve("https://files.rcsb.org/download/1D3Z.pdb", pdb_file)

# Download experimental NMR restraints if not present
rdc_file = "1d3z_mr.str"
if not os.path.exists(rdc_file):
    print("Downloading 1d3z NMR restraints...")
    subprocess.run(
        "curl -L -o 1d3z_mr.str.gz https://files.wwpdb.org/pub/pdb/data/structures/all/nmr_restraints/1d3z.mr.gz",
        shell=True,
    )
    subprocess.run("gunzip -f 1d3z_mr.str.gz", shell=True)

print("✅ Experimental data downloaded.")

## 2. Parsing the Experimental RDCs

The PDB restraints are formatted in older XPLOR/CNS syntaxes. We extract the N-H dipole couplings specifically from the bicelles alignment medium (`DipolarCouplings.HN-N.tbl`).

In [ ]:
import re

import numpy as np
import pandas as pd


def extract_nh_rdcs_from_xplor(filepath):
    """Extract N-H RDCs from the 1D3Z XPLOR/CNS restraint file."""
    with open(filepath) as f:
        content = f.read()

    # Extract the block containing N-H RDCs in bicelles
    start_idx = content.find("!!! DipolarCouplings.HN-N.tbl\n")
    end_idx = content.find("!!! DipolarCouplings.HN-CO.tbl\n", start_idx)

    if start_idx == -1:
        raise ValueError("Could not find HN-N RDC table in file")

    rdc_block = content[start_idx : end_idx if end_idx != -1 else len(content)]

    rdc_data = []
    lines = rdc_block.split("\n")

    current_res = None

    for line in lines:
        if "HN" in line and "and name" in line:
            parts = line.split(")")
            if len(parts) > 1:
                res_match = re.search(r"resid\s+(\d+)\s+and\s+name", line)
                if res_match:
                    current_res = int(res_match.group(1))
                nums = parts[1].strip().split()
                if nums and current_res is not None:
                    try:
                        val = float(nums[0])
                        rdc_data.append({"res_index": current_res, "D_exp": val})
                    except ValueError:
                        pass

    df = pd.DataFrame(rdc_data).drop_duplicates(subset=["res_index"])
    return df.sort_values("res_index").reset_index(drop=True)


df_rdc = extract_nh_rdcs_from_xplor(rdc_file)
print(f"Extracted {len(df_rdc)} experimental N-H RDCs from bicelles.")
df_rdc.head()

## 3. Calculating Synthetic RDCs with `synth-pdb`

We now feed the experimental 1D3Z structural coordinates into `synth_pdb` to map the biophysics. We calculate theoretical RDCs using an alignment tensor mathematically fitted to the 1D3Z structure.

In [ ]:
import biotite.structure.io.pdb as bpdb

# Load the PDB file into a Biotite AtomArray
pdb_struct = bpdb.PDBFile.read(pdb_file).get_structure(model=1)

# Extract N and H coordinates for the residues we have experimental data for
n_atoms = pdb_struct[pdb_struct.atom_name == "N"]
h_atoms = pdb_struct[(pdb_struct.atom_name == "H") | (pdb_struct.atom_name == "HN")]

# Create a mapping of residue index to N-H bond vector
nh_vectors = {}
for n in n_atoms:
    h = h_atoms[h_atoms.res_id == n.res_id]
    if len(h) > 0:
        # Vector points from N to H
        v = h.coord[0] - n.coord
        # Normalize
        v = v / np.linalg.norm(v)
        nh_vectors[n.res_id] = v

# Prepare matrices for SVD to find the Saupe order matrix (Alignment Tensor)
# We solve the overdetermined system: B * S_vec = D_exp
# Where S_vec are the 5 independent components of the trace-less, symmetric Saupe matrix
B = []
D = []

valid_res = []
for _idx, row in df_rdc.iterrows():
    res_id = int(row["res_index"])
    if res_id in nh_vectors:
        v = nh_vectors[res_id]
        x, y, z = v[0], v[1], v[2]

        # Coefficients for the 5 independent elements: Sxx, Syy, Sxy, Sxz, Syz
        # Using the relation D = D_max * sum_i sum_j v_i S_ij v_j
        row_B = [x**2 - z**2, y**2 - z**2, 2 * x * y, 2 * x * z, 2 * y * z]
        B.append(row_B)

        # D_max for N-H is approx -21700 Hz (assuming rigid bond length 1.04 A)
        # But we fit a scaled relative tensor directly to the experimental Hz
        D.append(row["D_exp"])
        valid_res.append(res_id)

B = np.array(B)
D = np.array(D)

# Solve using Singular Value Decomposition
S_vec, residuals, rank, s = np.linalg.lstsq(B, D, rcond=None)

# Reconstruct the 3x3 symmetric Saupe Matrix (S)
S = np.array(
    [
        [S_vec[0], S_vec[2], S_vec[3]],
        [S_vec[2], S_vec[1], S_vec[4]],
        [S_vec[3], S_vec[4], -(S_vec[0] + S_vec[1])],  # Trace is zero
    ]
)

# Diagonalize the matrix to find Principal Axes and Eigenvalues
eigenvalues, eigenvectors = np.linalg.eigh(S)

# Sort eigenvalues by absolute magnitude: |Szz| > |Syy| > |Sxx|
idx_sort = np.argsort(np.abs(eigenvalues))
Sxx, Syy, Szz = eigenvalues[idx_sort]

# Calculate Axial (Da) and Rhombic (R) components
# Da = Szz / 2
# R = (Sxx - Syy) / Szz
Da = Szz / 2.0
R = (Sxx - Syy) / Szz

print("SVD Tensor Fitting Complete!")
print(f"Fitted Axial Component (Da): {Da:.2f} Hz")
print(f"Fitted Rhombicity (R): {R:.3f}")

# Now calculate the theoretical RDCs using this exact fitted tensor
from synth_pdb.rdc import calculate_rdcs

# Note: The 'calculate_rdcs' function expects the PDB to be oriented such that
# its coordinates align with the principal axes of the tensor.
# Since we just found the principal axes (eigenvectors), we must rotate the structure!

# Create a rotation matrix from the eigenvectors
# eigenvectors[:, 2] is the strict Z-axis (associated with Szz), etc.
rotation_matrix = eigenvectors[:, idx_sort]

# Apply rotation to the Biotite structure
rotated_struct = pdb_struct.copy()
rotated_struct.coord = np.dot(rotated_struct.coord, rotation_matrix)

# Now standard calculation works perfectly:
theoretical_rdcs = calculate_rdcs(rotated_struct, da=Da, r=R)

df_rdc["D_calc"] = df_rdc["res_index"].map(lambda x: theoretical_rdcs.get(int(x), np.nan))
df_clean = df_rdc.dropna().copy()
print(f"Compiled {len(df_clean)} aligned overlapping RDCs for validation.")

## 4. Q-Factor Validation & Correlation

Let's calculate the Cornilescu Q-factor and standard Pearson's $R$ to evaluate the synthetic engine's performance.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

D_exp = df_clean["D_exp"].values
D_calc = df_clean["D_calc"].values

# Q-Factor Calculation (Cornilescu et al. 1998)
rms_diff = np.sqrt(np.mean((D_exp - D_calc) ** 2))
rms_exp = np.sqrt(np.mean(D_exp**2))
q_factor = rms_diff / rms_exp

# Pearson Correlation
r_val, _ = pearsonr(D_exp, D_calc)
r_sq = r_val**2

plt.figure(figsize=(8, 8))
plt.scatter(D_exp, D_calc, color="darkblue", alpha=0.7, edgecolor="white", s=80)

# Diagonal parity line
min_val = min(D_exp.min(), D_calc.min()) - 2
max_val = max(D_exp.max(), D_calc.max()) + 2
plt.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.5, label="Perfect Agreement")

plt.title("synth-pdb Validation: Synthetic vs Experimental RDCs", fontsize=14, fontweight="bold")
plt.xlabel("Experimental RDC $D_{HN}$ (Hz)", fontsize=12)
plt.ylabel("synth-pdb Calculated RDC $D_{HN}$ (Hz)", fontsize=12)

# Annotate stats
bbox_props = {"boxstyle": "round,pad=0.5", "fc": "white", "ec": "gray", "alpha": 0.9}
plt.text(
    min_val + 2,
    max_val - 5,
    f"Cornilescu Q-Factor: {q_factor:.3f}\nPearson $R^2$: {r_sq:.3f}",
    fontsize=12,
    bbox=bbox_props,
    family="monospace",
)

plt.grid(alpha=0.2)
plt.axis("equal")
plt.legend()
plt.show()

print("\n🏆 SCIENTIFIC CONCLUSION:")
print("=" * 60)
if q_factor < 0.25:
    print(
        f"SUCCESS! The computed Q-factor is {q_factor:.3f}, which is well below the <0.25 threshold."
    )
    print("This proves the `synth-pdb` orientation math and N-H vector extraction properly ")
    print(
        "recreates peer-reviewed measurable biology, demonstrating its validity as an NMR research tool."
    )
else:
    print(
        f"WARNING. Q-factor is {q_factor:.3f}. Further tuning of the alignment tensor may be required."
    )